In [12]:
from pathlib import Path
import pandas as pd

In [21]:
# Current working directory (where notebook runs)
current_path = Path().resolve()

# Go up manually depending on where you are
project_root = current_path.parent.parent   # adjust if needed

silver_file = project_root / "data" / "silver" / "dvf_paris_clean.csv"

output_path = project_root / "data" / "gold"

print(silver_file)

df = pd.read_csv(silver_file)

C:\Users\annan\Desktop\Quang Dat Doc\EFREI\DataSCience\Project-urban-data-explorer\data\silver\dvf_paris_clean.csv


In [14]:
df.columns = df.columns.str.strip()

In [15]:
df["valeur_fonciere"] = pd.to_numeric(df["valeur_fonciere"], errors="coerce")
df["surface_m2"] = pd.to_numeric(df["surface_m2"], errors="coerce")
df["prix_m2"] = pd.to_numeric(df["prix_m2"], errors="coerce")

In [16]:
gold_arrondissement = (
    df.groupby("arrondissement")
    .agg(
        nb_transactions=("valeur_fonciere", "count"),
        avg_price=("valeur_fonciere", "mean"),
        median_price=("valeur_fonciere", "median"),
        avg_price_m2=("prix_m2", "mean"),
        total_volume=("valeur_fonciere", "sum")
    )
    .reset_index()
    .sort_values("avg_price_m2", ascending=False)
)
display(gold_arrondissement.head())

,arrondissement,nb_transactions,avg_price,median_price,avg_price_m2,total_volume
5,6,845,1.078760e+06,675000.0,16449.554030,9.115523e+08
6,7,1019,1.468824e+06,911260.0,16068.949981,1.496731e+09
3,4,604,8.107473e+05,559000.0,14267.197117,4.896914e+08
7,8,749,1.216264e+06,850000.0,14211.861530,9.109814e+08
0,1,304,7.936766e+05,514681.0,13680.276910,2.412777e+08


In [17]:
gold_type = (
    df.groupby("type_local")
    .agg(
        nb_transactions=("valeur_fonciere", "count"),
        avg_price=("valeur_fonciere", "mean"),
        avg_surface=("surface_m2", "mean"),
        avg_price_m2=("prix_m2", "mean")
    )
    .reset_index()
    .sort_values("avg_price_m2", ascending=False)
)
display(gold_type)

,type_local,nb_transactions,avg_price,avg_surface,avg_price_m2
1,Maison,177,2.213703e+06,137.892655,15602.231692
0,Appartement,32075,6.070670e+05,52.637194,11114.384036


In [18]:
gold_monthly = (
    df.groupby(["annee", "mois"])
    .agg(
        nb_transactions=("valeur_fonciere", "count"),
        avg_price_m2=("prix_m2", "mean"),
        total_volume=("valeur_fonciere", "sum")
    )
    .reset_index()
    .sort_values(["annee", "mois"])
)
display(gold_monthly)

,annee,mois,nb_transactions,avg_price_m2,total_volume
0,2025,1,2366,11125.251694,1.469761e+09
1,2025,2,2192,11091.137407,1.289717e+09
2,2025,3,4033,11321.324475,2.592797e+09
3,2025,4,1779,10570.925677,9.594402e+08
4,2025,5,2244,11147.423539,1.358203e+09
5,2025,6,3010,10844.721955,1.803711e+09
6,2025,7,3907,11345.786637,2.503088e+09
7,2025,8,1390,10706.561760,7.385040e+08
8,2025,9,3446,11092.910216,2.141623e+09
9,2025,10,2931,11032.855679,1.786682e+09


In [19]:
gold_distribution = df[[
    "arrondissement",
    "type_local",
    "surface_m2",
    "prix_m2",
    "valeur_fonciere"
]].dropna()

In [20]:
gold_type.to_csv(
    output_path / "dvf_paris_by_type.csv",
    index=False, encoding="utf-8-sig"
)
gold_arrondissement.to_csv(
    output_path / "dvf_paris_by_arrondissement.csv",
    index=False, encoding="utf-8-sig"
)
gold_monthly.to_csv(
    output_path / "dvf_paris_monthly.csv",
    index=False, encoding="utf-8-sig"
)
gold_distribution.to_csv(
    output_path / "dvf_paris_distribution.csv",
    index=False, encoding="utf-8-sig"
)